# 22 — Core Contracts & Agent Catalog

Demonstrates the **typed contract layer** introduced in the architectural audit (Sprints 1–4):

| Module | What changed |
|---|---|
| `ravi.core.contracts` | `ToolCallRequest`, `ToolExecutionResult`, `CanonicalMessage`, `EventEnvelope[T]` |
| `ravi.core.agent_catalog` | `AgentCatalog`, `ResourceSpec`, `ResourceType` — replaces god-object registry |
| `ravi.core.agents._tool_execution` | `ToolExecutionContext` — collapses 22-param function signature |
| `ravi.core.messages` | Canonical `tool_call_id`, `UsageStats` auto-totalling, `from_tool_result()` simplified |
| `ravi.core.middleware` | Rate-limiter lock fix — sleep now outside the lock |

**Prerequisites**: no external services needed — all cells are offline.

---
## 1. Tool contracts — `ToolCallRequest` & `ToolExecutionResult`

`ravi.core.contracts._tool` provides immutable, typed request/result objects
that replace bare `dict` passing across the tool execution boundary.

In [1]:
from ravi.core.contracts import ToolCallRequest, ToolExecutionResult
from ravi.core.messages.content import TextBlock

# Build a typed request — all fields validated at construction time
req = ToolCallRequest(
    name="calculator",
    arguments={"expression": "2 ** 10"},
    agent_name="DemoBot",
    step=3,
)
print(f"call_id : {req.call_id!r}   (auto-generated UUID)")
print(f"name    : {req.name}")
print(f"args    : {req.arguments}")
print(f"frozen  : {req.model_config.get('frozen')}  ← immutable after creation")

# Build a typed result
result = ToolExecutionResult(
    call_id=req.call_id,
    name=req.name,
    content=[TextBlock(text="1024")],
    duration_ms=12.4,
)
print(f"\nresult.text → {result.text!r}")
print(f"is_error    → {result.is_error}")

call_id : '945a9e84-606c-4a87-be01-a806bc9e9960'   (auto-generated UUID)
name    : calculator
args    : {'expression': '2 ** 10'}
frozen  : True  ← immutable after creation

result.text → '1024'
is_error    → False


---
## 2. Message contracts — `CanonicalMessage` & `MessageRole`

Provider-neutral message format used when passing messages across the agent ↔ LLM boundary.
Encoders translate `CanonicalMessage` → OpenAI / Anthropic wire format.

In [2]:
from ravi.core.contracts import CanonicalMessage, MessageRole, ToolCallSpec
from ravi.core.messages.content import TextBlock

# A user message
user_msg = CanonicalMessage(
    role=MessageRole.USER,
    content=[TextBlock(text="What is 2+2?")],
)

# An assistant message that includes a tool call
assistant_msg = CanonicalMessage(
    role=MessageRole.ASSISTANT,
    tool_calls=[
        ToolCallSpec(call_id="tc-001", name="calculator", arguments={"expression": "2+2"})
    ],
)

# A tool result message
tool_msg = CanonicalMessage(
    role=MessageRole.TOOL,
    content=[TextBlock(text="4")],
    tool_call_id="tc-001",
    name="calculator",
)

for m in [user_msg, assistant_msg, tool_msg]:
    tc = m.tool_calls[0].name if m.tool_calls else "-"
    text = m.content[0].text if m.content else "-"
    print(f"role={m.role.value:<10}  tool={tc:<12}  text={text!r}")

role=user        tool=-             text='What is 2+2?'
role=assistant   tool=calculator    text='-'
role=tool        tool=-             text='4'


---
## 3. `tool_call_id` — canonical field across all message types

Before the audit, `ToolCallMessage` used `.id` while `ToolExecutionResultMessage`
used `.tool_call_id` — two names for the same concept.

Now `ToolCallMessage.tool_call_id` is a property alias for `.id`, and all
provider encoders use `.tool_call_id` consistently.

In [3]:
from ravi.core.messages.client_messages import ToolCallMessage, ToolExecutionResultMessage

tc = ToolCallMessage(id="abc-123", name="calculator", arguments={"expression": "1+1"})

print(f"tc.id           → {tc.id!r}")
print(f"tc.tool_call_id → {tc.tool_call_id!r}   ← new canonical property")
print(f"Same object?    → {tc.id is tc.tool_call_id}")

tr = ToolExecutionResultMessage(
    tool_call_id="abc-123",
    content=[TextBlock(text="2")],
)
print(f"\ntr.tool_call_id → {tr.tool_call_id!r}   (always existed)")
print(f"Match?          → {tc.tool_call_id == tr.tool_call_id}")

tc.id           → 'abc-123'
tc.tool_call_id → 'abc-123'   ← new canonical property
Same object?    → True

tr.tool_call_id → 'abc-123'   (always existed)
Match?          → True


---
## 4. `UsageStats` — auto-total when provider omits it

Some providers return `prompt_tokens` and `completion_tokens` but leave
`total_tokens=0`. The new `_normalize_total` validator fills it in automatically.

In [4]:
from ravi.core.messages.base_message import UsageStats

# Provider forgot to set total_tokens
stats = UsageStats(prompt_tokens=500, completion_tokens=200, total_tokens=0)
print(f"prompt      : {stats.prompt_tokens}")
print(f"completion  : {stats.completion_tokens}")
print(f"total       : {stats.total_tokens}  ← auto-computed (was 0)")
assert stats.total_tokens == 700

# If provider already set total, it is left unchanged
stats2 = UsageStats(prompt_tokens=100, completion_tokens=50, total_tokens=175)
print(f"\ntotal (explicit) : {stats2.total_tokens}  ← not overwritten")

prompt      : 500
completion  : 200
total       : 700  ← auto-computed (was 0)

total (explicit) : 175  ← not overwritten


---
## 5. `AgentCatalog` — unified resource registry

Replaces the old 726-line `AgentCatalogRegistry` god-object with a clean,
single-entry-point `AgentCatalog` backed by `ResourceSpec` metadata.

Key improvements:
- Collision detection: duplicate FQN raises `ValueError` instead of silently overwriting
- `check_permission()` now returns `bool` (old version returned `None` on no-match)
- `get_tool()` checks `spec.resource_type` instead of `isinstance(instance, BaseTool)`

In [5]:
from ravi.core.agent_catalog import AgentCatalog, ResourceSpec, ResourceType

catalog = AgentCatalog()  # default_catalog="main"

# ResourceSpec namespace must be "{catalog}.{schema}" so the short-name
# resolver can find it. The default catalog is "main"; default schema is "default".
spec = ResourceSpec(
    name="search",
    namespace="main.default",  # ← catalog.schema — enables short-name lookup
    resource_type=ResourceType.TOOL,
    description="Web search tool",
    tags=["web", "search"],
)
print(f"FQN: {spec.fqn}")   # main.default.search

fake_tool = object()
catalog.register(spec, fake_tool)
print(f"Registered: {spec.fqn}")

# Short-name lookup works when namespace == "{default_catalog}.{schema}"
found = catalog.get_tool("search")
print(f"get_tool('search') → {found is fake_tool}")

# Full FQN lookup also works
found_fqn = catalog.get_tool("main.default.search")
print(f"get_tool('main.default.search') → {found_fqn is fake_tool}")

# Collision detection — duplicate FQN raises ValueError
try:
    catalog.register(spec, object())
except ValueError as e:
    print(f"Collision detected: {e}")

# check_permission(principal, target_fqn, privilege="execute")
# With no grants configured → open access (returns True)
result = catalog.check_permission("any_agent", "main.default.search")
print(f"check_permission (open) → {result!r}  ({type(result).__name__})")

# Add a grant, then check a denied principal
catalog.grant_privilege("execute", "main.default.*", "trusted_agent")
allowed = catalog.check_permission("trusted_agent", "main.default.search")
denied  = catalog.check_permission("untrusted",     "main.default.search")
print(f"trusted_agent → {allowed}  |  untrusted → {denied}")


FQN: main.default.search
Registered: main.default.search
get_tool('search') → True
get_tool('main.default.search') → True
Collision detected: Catalog collision: 'main.default.search' is already registered. Use unregister() first if replacement is intended.
check_permission (open) → True  (bool)
trusted_agent → True  |  untrusted → False


---
## 6. `ResourceType` enum — all resource categories

Every resource registered in the catalog carries a typed `ResourceType`.
The catalog uses this instead of `isinstance()` checks.

In [6]:
from ravi.core.agent_catalog import ResourceType

for rt in ResourceType:
    print(f"  {rt.value}")

  tool
  skill
  memory
  context
  checkpoint
  mcp_tool
  model


---
## 7. `ToolExecutionContext` — collapsing the 22-parameter function

`execute_tool_direct()` previously took 22 keyword arguments.  
Now the agent builds a `ToolExecutionContext` once per call and passes it as a single object.  
This makes the call sites readable and the dataclass easy to test in isolation.

In [7]:
from ravi.core.agents._tool_execution import ToolExecutionContext
import dataclasses

fields = [f.name for f in dataclasses.fields(ToolExecutionContext)]
print(f"ToolExecutionContext has {len(fields)} fields:")
for f in fields:
    print(f"  {f}")

ToolExecutionContext has 15 fields:
  agent_name
  run_id
  tool_timeout
  tool_retry_policy
  verbose
  hooks
  middleware_pipeline
  catalog
  tools
  execution_context
  tool_search_name
  activate_tool_names_cb
  skill_manager
  runtime
  agent_id


---
## 8. Rate limiter middleware — lock fix

**Before**: `asyncio.sleep()` was called *inside* `async with self._lock`.  
All concurrent callers would queue behind a single sleep, eliminating any parallelism.

**After**: the lock is released before sleeping. Waiters sleep concurrently and  
each re-acquires the lock afterward to consume a token.

In [8]:
import asyncio, time
from ravi.core.middleware.builtins.rate_limiter import RateLimiterMiddleware
from ravi.core.middleware.base import MiddlewareContext, MiddlewareStage
from ravi.core.execution.context import ExecutionContext

# Allow only 2 requests per second
limiter = RateLimiterMiddleware(max_rate=2.0, per_seconds=1.0)

ctx = MiddlewareContext(
    stage=MiddlewareStage.LLM_CALL,
    agent_name="test",
    run_id="r1",
)

async def run_parallel(n: int) -> float:
    t0 = time.monotonic()
    await asyncio.gather(*[limiter.before(ctx) for _ in range(n)])
    return time.monotonic() - t0

elapsed = await run_parallel(4)   # 4 requests, 2/s budget → ~1s wait for the extra 2
print(f"4 requests through a 2/s limiter took {elapsed:.2f}s")
print("(expected ≥ 1.0s — extra requests sleep concurrently, not serially)")

4 requests through a 2/s limiter took 1.00s
(expected ≥ 1.0s — extra requests sleep concurrently, not serially)


---
## 9. `EventEnvelope` — backward-compat re-export

`core.contracts.EventEnvelope[T]` is the generic typed version for call-site type safety.  
`shared.events.envelope.EventEnvelope` is the bus-level version (untyped payload) used  
by the Redis Streams backbone. Both share the same field structure.

In [9]:
from ravi.core.contracts import EventEnvelope as CoreEnvelope
from ravi.shared.events.envelope import EventEnvelope as BusEnvelope

# Core generic version — payload is typed
core_ev = CoreEnvelope[dict](event_type="test.event", payload={"x": 1})
print(f"Core envelope  : event_type={core_ev.event_type!r}  payload={core_ev.payload}")

# Bus version — also has trace_context (Sprint 8 addition)
bus_ev = BusEnvelope(event_type="test.event", payload={"x": 1})
print(f"Bus envelope   : trace_context={bus_ev.trace_context!r}  (empty until publish)")
print(f"stream_key()   : {bus_ev.stream_key()!r}")

Core envelope  : event_type='test.event'  payload={'x': 1}
Bus envelope   : trace_context={}  (empty until publish)
stream_key()   : 'events:test.event'


---
## 10.  — LLM clients in the catalog

The catalog can hold any registered model client, not just tools.
This enables **multi-model setups** where different agents or tasks
select the right LLM by name — without passing clients around as constructor args.

Use  to register;  to retrieve.


In [ ]:
from ravi.core.agent_catalog import AgentCatalog, ResourceType

model_catalog = AgentCatalog()

class FakeClient:
    def __init__(self, model): self.model = model
    def __repr__(self): return f"FakeClient({self.model!r})"

model_catalog.register_model("gpt4o",  FakeClient("gpt-4o"))
model_catalog.register_model("claude", FakeClient("claude-sonnet-4-6"))
model_catalog.register_model("cheap",  FakeClient("gpt-4o-mini"))

# Typed accessor — new in this sprint
for name in ["gpt4o", "claude", "cheap"]:
    client = model_catalog.get_model(name)
    print(f"  {name:<8} -> {client}")

# resolve() also works for any resource type
smart = model_catalog.resolve("claude")
print(f"\nresolved 'claude' -> {smart}")

# Real usage:
# from ravi.integrations.llm.openai.openai_client import OpenAIClient
# catalog.register_model("fast",  OpenAIClient(model="gpt-4o-mini"))
# catalog.register_model("smart", OpenAIClient(model="gpt-4o"))
# client = catalog.get_model("smart")  # swap LLMs without touching agent code


  gpt4o    -> FakeClient('gpt-4o')
  claude   -> FakeClient('claude-sonnet-4-6')
  cheap    -> FakeClient('gpt-4o-mini')

resolved 'claude' -> FakeClient('claude-sonnet-4-6')


: 